# Setup

In [1]:
# Load necessary packages. 
import os
import re
import sys
import numpy as np
import pandas as pd

# Define current directory. 
parent_dir = "./"

# Define folders to hold all results. 
folders = sorted([
    "results/beta_0_rep1", "results/beta_0_rep2", "results/beta_0_rep3", "results/beta_0_rep4", "results/beta_0_rep5",
    "results/beta_2_rep1", "results/beta_2_rep2", "results/beta_2_rep3", "results/beta_2_rep4", "results/beta_2_rep5"
])

# Define P-value threshold. 
padj_thresh = 0.05

# Helper functions

In [2]:
# Defines whether each site has a true change or not. 
def classify_true_change(true_zeta):
    if pd.isna(true_zeta):
        return "no_change"
    if true_zeta > 1.5:
        return "increased"
    if true_zeta < 0.66:
        return "decreased"
    return "no_change"

# Used to extract the correct parameters from each simulation run name. 
def parse_sim_name(folder):
    m = re.match(r"^beta_(-?\d+(?:\.\d+)?)_rep(\d+)$", folder)
    return({"beta_sd": float(m.group(2)), "rep": int(m.group(3))})

# Creates a confusion matrix that is used to calculate sensitivity and precision. 
def compute_confusion(truth_df, pred_df, pred_col = "change_predict"):
    # Merge the true differential results and the predicted differential results. 
    merged = truth_df[["window_name", "change_true"]].merge(pred_df[["window_name", pred_col]],
                                                            on="window_name", how="left")
    
    # Create an empty predictions column. 
    merged[pred_col] = merged[pred_col].fillna("no_change")

    # Extract the truth and the predictions, convert to string. 
    t = merged["change_true"]
    p = merged[pred_col]

    # Calculate the number of true positives, false negatives, etc. 
    fp = ((t == "no_change") & (p != "no_change")).sum()
    fn = ((t != "no_change") & (p == "no_change")).sum()
    tp = ((t == "increased") & (p == "increased")).sum() + ((t == "decreased") & (p == "decreased")).sum()
    fp = fp + ((t == "increased") & (p == "decreased")).sum() + ((t == "decreased") & (p == "increased")).sum()

    # Calculate precision and sensitivity. 
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan

    return({"tp": int(tp), "fp": int(fp), "fn": int(fn),
            "precision": precision, "sensitivity": sensitivity,
            "n_truth": int(len(truth_df))})

# Load and process the true simulation parameters. 
def load_truth(sim_dir: str):
    true_sites = pd.read_csv(f"{sim_dir}/true_values/binding_metrics.tsv", sep="\t")
    # Load in the skipper windows. 
    skipper_windows = pd.read_csv(f"{sim_dir}/flipper/intermediates/window_counts_INf.tsv", sep = "\t")
    # Select skipper windows.
    true_sites = true_sites[
        true_sites["window_name"].isin(skipper_windows["window_name"])
    ]
    true_sites["change_true"] = true_sites["trueZeta"].apply(classify_true_change)
    true_sites["window_name"] = true_sites["window_name"]
    return true_sites

# Load and process the 
def load_flipper_predictions(sim_dir, padj_thresh):
    # Load up the data. 
    path = os.path.join(sim_dir, "flipper", "tables", "differential_windows.tsv")
    df = pd.read_csv(path, sep="\t")
    df["change_predict"] = np.where(
        (df["padj"] < padj_thresh) & (df["log2FoldChange"] > 0), "increased",
        np.where((df["padj"] < padj_thresh) & (df["log2FoldChange"] < 0), "decreased","no_change")
    )
    return(df[["window_name", "change_predict"]])


def load_adhoc_skipper_predictions(sim_dir):
    # Load up the reproducible skipper windows from each condition. 
    ctrl_path = os.path.join(sim_dir, "reproducible_enriched_windows", "control.reproducible_enriched_windows.tsv.gz")
    trt_path  = os.path.join(sim_dir, "reproducible_enriched_windows", "treatment.reproducible_enriched_windows.tsv.gz")
    ctrl = pd.read_csv(ctrl_path, sep="\t")
    trt  = pd.read_csv(trt_path, sep="\t")

    # Convert to sets. 
    control_windows = set(ctrl["window_name"])
    treatment_windows = set(trt["window_name"])

    # Find sites only in control and label as decreased.
    only_in_control = ctrl.loc[~ctrl["window_name"].isin(treatment_windows), ["window_name"]].copy()
    only_in_control["change_predict"] = "decreased"

    # Find sites only in treatment and label as increased.
    only_in_treatment = trt.loc[~trt["window_name"].isin(control_windows), ["window_name"]].copy()
    only_in_treatment["change_predict"] = "increased"

    # Find sites in both and label as no change. 
    in_both = pd.DataFrame({"window_name": sorted(control_windows.intersection(treatment_windows))})
    in_both["change_predict"] = "no_change"

    # Combine into a single dataset. 
    out = pd.concat([only_in_control, only_in_treatment, in_both], ignore_index=True)
    return(out)

def load_diffskip_predictions(sim_dir):
    # Load "up" and "down" regulated reproducible sites from diff-skipper. 
    up_path   = os.path.join(sim_dir, "diff_skipper", "reproducible_enriched_windows", "ctr-trt.reproducible_enriched_windows.tsv")
    down_path = os.path.join(sim_dir, "diff_skipper", "reproducible_enriched_windows", "trt-ctr.reproducible_enriched_windows.tsv")
    up = pd.read_csv(up_path, sep="\t")
    down = pd.read_csv(down_path, sep="\t")

    # Define increased and decreased sites. 
    up = up[["window_name"]]
    down = down[["window_name"]]
    up["change_predict"] = "increased"
    down["change_predict"] = "decreased"

    # Concatenate and return. 
    out = pd.concat([up, down], ignore_index=True)
    return(out)

# Summarize results

In [3]:
# Initialize empty lists to hold 
flipper_rows = []
adhoc_rows = []
diffskip_rows = []

# A for loop that goes through all folders and calculates performance for all programs. 
for fld in folders:
    sim_dir = os.path.join(parent_dir, fld)

    truth = load_truth(sim_dir)

    # flipper.
    flipper_pred = load_flipper_predictions(sim_dir, padj_thresh=padj_thresh)
    cm = compute_confusion(truth_df=truth, pred_df=flipper_pred, pred_col="change_predict")
    cm["folder"] = fld
    flipper_rows.append(cm)

    # Adhoc skipper.
    adhoc_pred = load_adhoc_skipper_predictions(sim_dir)
    cm = compute_confusion(truth_df=truth, pred_df=adhoc_pred, pred_col="change_predict")
    cm["folder"] = fld
    adhoc_rows.append(cm)

    # DiffSkip.
    diffskip_pred = load_diffskip_predictions(sim_dir)
    cm = compute_confusion(truth_df=truth, pred_df=diffskip_pred, pred_col="change_predict")
    cm["folder"] = fld
    diffskip_rows.append(cm)

# Convert lists to dataframes. 
flipper_df = pd.DataFrame(flipper_rows)
adhoc_df = pd.DataFrame(adhoc_rows)
diffskip_df = pd.DataFrame(diffskip_rows)

# Save dataframes. 
flipper_df.to_csv("results/flipper_results_summary.tsv", sep="\t")
adhoc_df.to_csv("results/adhoc_results_summary.tsv", sep="\t")
diffskip_df.to_csv("results/diffskip_results_summary.tsv", sep="\t")

# Combine all dataframes together. 
combined = pd.concat([pd.DataFrame(flipper_rows), pd.DataFrame(adhoc_rows), pd.DataFrame(diffskip_rows)],
                     ignore_index=True)

# Save combined dataframes. 
combined.to_csv("results/all_methods_summary.tsv", sep="\t", index=False)